In [2]:
import torch
from pathlib import Path

from eval import inference, top_k_accuracy
from model import LeagueDraftModel
from vocabulary import Vocabulary
from data import load_matches, ChampionDataset
from torch.utils.data import DataLoader
from draft_constraints import mask_logits

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

BATCH_SIZE = 2048

In [3]:
start = Path.cwd().resolve()

PROJECT_ROOT = next(
    path
    for path in (start, *start.parents)
    if (path / '.git').exists()
)

CHECKPOINT_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'checkpoints'
DATA_DIRECTORY = PROJECT_ROOT / 'data' 

In [4]:
checkpoint = torch.load(CHECKPOINT_DIRECTORY / 'best_model.pth', device, weights_only=True)
state_dict = checkpoint['model_state_dict']
champ_dict = checkpoint['riotid_to_name']

vocab = Vocabulary(champ_dict)
model = LeagueDraftModel(len(vocab), vocab.mask_id)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

encoded_matches = load_matches(DATA_DIRECTORY / 'league_data.db', vocab)

test_data = ChampionDataset(encoded_matches, vocab.mask_id)

In [5]:
eval_loader = DataLoader(
    test_data, 
    batch_size = BATCH_SIZE , 
    shuffle=False,
    pin_memory=(device.type=='cuda')
)

In [6]:
print(top_k_accuracy(eval_loader, model, device, 5), top_k_accuracy(eval_loader, model, device, 10))

0.4075501752150366 0.5829882128066263


In [21]:
# does the model use any draft context when picking? calculate top_5_accuracy with random context
count = 0
valid_examples = 0
model.eval()
with torch.inference_mode(): 
    for picks, bans, target in eval_loader:

        picks = picks.to(device, non_blocking=(device.type == 'cuda'))
        bans = bans.to(device, non_blocking=(device.type == 'cuda'))
        target = target.to(device, non_blocking=(device.type == 'cuda'))
        
        # find where the mask id is
        masks = (picks == vocab.mask_id).to(device, non_blocking=(device.type == 'cuda'))

        # generate a random draft
        rand_picks = torch.randint_like(picks, 0, len(vocab)-2, device=device)

        # set the locations that should have a mask id to mask_id
        rand_picks[masks] = vocab.mask_id

        # keep only the matches that have not randomly generated the target in the context
        is_valid = (rand_picks != target.unsqueeze(1)).all(1)
        picks = picks[is_valid]
        rand_picks = rand_picks[is_valid]
        bans = bans[is_valid]
        target = target[is_valid]

        logits = model(rand_picks)
        logits = mask_logits(picks, bans, logits)
    
        # returns value indices pairs, only care about index
        _, preds = torch.topk(logits, k=5, dim=1)

        valid_examples += is_valid.shape[0]
        count += torch.sum(torch.sum(preds == target.unsqueeze(1), dim=1))

In [22]:
# top_5 accuracy with random context
print(count.item()/valid_examples)

0.3056387384517362


In [10]:
game = ['malphite', 'diana', 'ahri', 'masked', 'lulu', 'darius', 'warwick', 'orianna', 'ezreal', 'karma']

print(inference(model, game, vocab, device, k=5))

['Yunara', 'Jinx', 'Aphelios', 'Zeri', 'Tristana']
